In [1]:
import os
os.chdir("..")

In [2]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [3]:
from src.cnnClassifer.constants.paths import CONFIG_FILE_PATH, PARAMS_FILE_PATH
from src.cnnClassifer.utils.common import read_yaml, create_directories

In [4]:
class ConfiguartionManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        
        config = self.config.data_ingestion
        data_ingestion_config = DataIngestionConfig(
            root_dir=Path(config.root_dir),
            source_URL=config.source_URL,
            local_data_file=Path(config.local_data_file),
            unzip_dir=Path(config.unzip_dir)
        )
        return data_ingestion_config
    

In [5]:
import os
import urllib.request as request
import zipfile
from cnnClassifer.utils.common import get_size
from cnnClassifer import logger

In [6]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config
        
        create_directories([self.config.root_dir])
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url =self.config.source_URL,
                filename = self.config.local_data_file
            )
        logger.info(f"{filename} downloaded with following info: \n{headers}")


    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(file=self.config.local_data_file, mode="r") as zip_ref:
            zip_ref.extractall(unzip_path)

            

In [7]:
try:
    config= ConfiguartionManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e


[2026-06-21 03:39:43,569: INFO: common: YAML file: config\config.yaml loaded successfully]
[2026-06-21 03:39:43,572: INFO: common: YAML file: params.yaml loaded successfully]
[2026-06-21 03:39:43,576: INFO: common: Directory created at: artifacts]
[2026-06-21 03:39:43,576: INFO: common: Directory created at: artifacts\data_ingestion]
[2026-06-21 03:39:49,788: INFO: 4227262218: artifacts\data_ingestion\data.zip downloaded with following info: 
Connection: close
Content-Length: 6489735
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "c6fc7a70c409eea39f5075968c0f83062e0302eea6d329a17eae5ab589f5c535"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: 5830:7BC2C:15A93D5:186A22A:6A370FA1
Accept-Ranges: bytes
Date: Sat, 20 Jun 2026 22:39:44 GMT
Via: 1.1 varnish
X-Served-By: cache-mrs10565-MRS
X-Cache: